# 07_02 Optimisers raced: which way down, and how big a step?

Every optimiser uses the same gradients. They differ only in what step they take with them. This notebook
trains the same small network on Reuters newswires five times, changing nothing but the optimiser and its
learning rate, and measures how far each gets in 300 steps.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-07-how-a-network-learns", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'nltk': 'nltk',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words', 'reuters']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import math
import os
import time
import torch
import matplotlib
import matplotlib.pyplot as plt
import newswire
from nlpcheck import ask, guess, reveal, check_07_02

torch.set_num_threads(4)
data = newswire.load_split()   # built at session start; a few seconds if not
X_train, y_train, X_test, y_test = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
print(len(X_train), "training and", len(X_test), "test newswires,", len(newswire.TOPICS), "topics:", newswire.TOPICS)

## 1. Recall

**r3.** In gradient descent, if a weight's gradient is positive, what happens to the weight?
(a) it gets smaller, (b) it gets larger, (c) it stays the same

**r4.** What does `loss.backward()` do? (a) takes one step of gradient descent, (b) fills in `.grad` for
every tensor that requires it, and changes no weight, (c) resets the gradients to zero

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. The data and the network

**Reuters-21578** is a collection of newswires from 1987, each tagged with topics. It is already in the
lab (the ApteMod split, as packaged by NLTK); the original collection is at the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/137/reuters+21578+text+categorization+collection)
under CC BY 4.0. `newswire.py` keeps the newswires with exactly one of six common topics, and turns each into
its first 50 words as vocabulary ids.

In [ ]:
print(data["raw_test"][0][:400])
print("encoded:", X_test[0][:20].tolist(), "...")

The network for the race is `newswire.BagClassifier`: it looks up a 64-number vector for each word, averages
them, and a linear layer turns the average into six topic scores. It trains in a few seconds, which is what a race
needs. The function below trains it for 300 steps of 64 newswires each, with any optimiser, and returns the
loss curve and the test accuracy. The seeds are fixed, so every optimiser sees the same batches in the same
order and starts from the same weights.

In [ ]:
def race(make_optimiser, steps=300, batch_size=64):
    torch.manual_seed(0)
    model = newswire.BagClassifier()
    opt = make_optimiser(model.parameters())
    if opt is None:
        return None
    g = torch.Generator().manual_seed(1)
    losses = []
    for _ in range(steps):
        batch = torch.randint(0, len(X_train), (batch_size,), generator=g)
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(model(X_train[batch]), y_train[batch])
        loss.backward()
        opt.step()
        losses.append(loss.item())
    acc = newswire.evaluate(model, X_test, y_test)["accuracy"]
    final = sum(losses[-50:]) / 50      # the mean of the last 50 steps, less noisy than one step
    return losses, final, acc

results, curves = {}, {}
def record(name, make_optimiser):
    t = time.time()
    out = race(make_optimiser)
    if out is None:
        print(f"{name:14} no optimiser yet: finish the line above")
        return
    losses, final, acc = out
    results[name] = {"final_loss": final, "test_accuracy": acc}
    curves[name] = losses
    print(f"{name:14} final loss {final:.3f}   test accuracy {acc:.3f}   ({time.time() - t:.1f} s)")

## 3. Plain stochastic gradient descent

**SGD** takes a mini-batch, computes the gradient on it, and steps `lr * gradient` downhill. With 64 newswires
per step instead of all 5,352, each step is a noisy estimate of the true gradient, and far cheaper. Each race
takes a few seconds.

In [ ]:
record("sgd_0.1", lambda p: torch.optim.SGD(p, lr=0.1))
record("sgd_1.0", lambda p: torch.optim.SGD(p, lr=1.0))

Ten times the learning rate, and the loss after 300 steps falls from about 0.76 to about 0.42: at 0.1 the
steps were simply too small to get far. So why not go much bigger? Predict what learning rate 50 does:
(a) gets there even faster, (b) about the same as 1.0, (c) the loss becomes nan (not a number).

In [ ]:
guess("lr_50", None)

In [ ]:
record("sgd_50.0", lambda p: torch.optim.SGD(p, lr=50.0))
reveal("lr_50", "c" if math.isnan(results["sgd_50.0"]["final_loss"]) else "a")

The loss is `nan` and the accuracy is 0.505, the share of the most common topic: the network ends up
predicting one topic for everything. Each step overshot the bottom of the valley by more than it started
from, so the next gradient was larger, and the next step larger still, until the numbers overflowed. The
book's warning, measured: a learning rate that is too large does not reach the minimum sooner; it diverges.

## 4. Momentum: remembering the last step

**Momentum** keeps a running velocity: each step is the new gradient plus 0.9 of the previous step. On a
long, steady slope the steps add up; in a direction where the gradient keeps flipping sign, they cancel.
Predict: momentum 0.9 at learning rate 0.1 will finish closest to which plain SGD run?
(a) SGD at 0.1, (b) SGD at 1.0, (c) neither

In [ ]:
guess("momentum_like", None)

## 5. Your turn: run the momentum race

Complete the line: `torch.optim.SGD` with `lr=0.1` and `momentum=0.9`.

In [ ]:
record("momentum_0.1", lambda p: None)   # YOUR CODE HERE: replace None with the SGD optimiser with momentum
if "momentum_0.1" in results:
    close = abs(results["momentum_0.1"]["final_loss"] - results["sgd_1.0"]["final_loss"]) < 0.08
    reveal("momentum_like", "b" if close else "a")

SGD at 1.0. With a steady gradient `g`, the velocity settles where `v = g + 0.9 v`, which is `v = 10 g`:
momentum 0.9 multiplies the effective step by `1 / (1 - 0.9) = 10`. The difference from simply raising the
learning rate is that the velocity is an average, so a gradient that jitters from batch to batch is smoothed
before it moves the weights.

## 6. Adam: a learning rate for every weight

**Adam** keeps two running averages for every single weight: of its gradient (like momentum) and of its
gradient squared. It divides one by the square root of the other, so a weight whose gradient is always tiny
(a rare word's embedding) still takes a sensible step, and one whose gradient is huge does not explode.
**AdamW** is the same with weight decay applied the corrected way, and is the default in most current
training code.

In [ ]:
record("adam_0.002", lambda p: torch.optim.Adam(p, lr=0.002))
record("adam_0.01", lambda p: torch.optim.Adam(p, lr=0.01))
record("adamw_0.01", lambda p: torch.optim.AdamW(p, lr=0.01))

At learning rate 0.01, a final loss of about 0.04 and test accuracy near 0.97, ten times lower than anything
SGD reached in the same 300 steps. At 0.002, Adam's loss is about 0.34: better than SGD, and eight times worse
than Adam at 0.01. Adapting every weight's step does not remove the learning rate; it sets the scale all the
adaptation happens around. This is why the chapter ends "overall, Adam is the best choice", and why you will see AdamW in
nearly every training script you read. It is not magic: on a very large, well-tuned job, SGD with momentum
can still match it. It is the optimiser that needs the least tuning to work.

The loss curves, side by side (the diverged run is left out, because nan cannot be drawn):

In [ ]:
plt.figure(figsize=(9, 4))
for name, losses in curves.items():
    if not math.isnan(results[name]["final_loss"]):
        smooth = [sum(losses[max(0, i - 19):i + 1]) / len(losses[max(0, i - 19):i + 1]) for i in range(len(losses))]
        plt.plot(smooth, label=name)
plt.xlabel("step"); plt.ylabel("training loss (mean of last 20 steps)"); plt.legend(); plt.tight_layout()
plt.savefig("out/07_02_race.png", dpi=90)
plt.show()

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/07_02_race.json", "w"), indent=1)
check_07_02()

## 7. Exit ticket

**x2.** What does Adam do that plain SGD does not? (a) gives every weight its own effective step size, from
running averages of its gradient and squared gradient, (b) computes exact gradients over the whole dataset,
(c) never needs a learning rate

In [ ]:
ask("x2", "")

Explain it back: why did learning rate 50 end in nan rather than in the fastest training?

*Your explanation:* 